In [3]:
import base64
import json
import os
import random
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Dict

定义图像编码函数

- 4o 系列：请使用 Base64 编码将图片转换为字符串格式。
- QwenVL 系列：可以直接发送原始图片数据，无需进行编码。


In [4]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

定义 QA 对


In [5]:
class QA_Pair(BaseModel):
    question: str
    answer: str

##### 类定义：`Cmanager`

`Cmanager`类是一个用于管理网络拓扑图像和相关操作的类。它提供了初始化客户端、设置拓扑图像路径、提取实体、构建问答对以及保存数据到 JSON 文件的功能。
属性：

- `client`: 用于与 API 通信的客户端对象。
- `topology_image_path`: 存储拓扑图像文件路径的字符串。
  方法：

1. `__init__(self, api_base: str, api_key: str)`：
   - 初始化方法，接收两个参数：`api_base`（API 的基础 URL）和`api_key`（API 的密钥）。
   - 创建客户端实例并存储在`self.client`中。
   - 初始化`topology_image_path`为空字符串。
2. `get_client(self, base_url, api_key)`：
   - 私有方法，用于创建并返回一个`OpenAI`客户端实例。
   - 参数：`base_url`和`api_key`。
3. `set_topology_image(self, image_path: str)`：
   - 设置拓扑图像文件路径。
   - 参数：`image_path`（图像文件的路径）。
4. `extract_entities(self) -> List[str]`：
   - 提取拓扑图像中的网络实体。
   - 检查`topology_image_path`是否已设置，如果没有，则抛出`ValueError`。
   - 将图像编码为 base64 格式，并构建系统内容字符串。
   - 使用`gpt-4o-mini`模型创建聊天完成请求，提取实体并返回实体列表。
5. `build_qa_pairs(self, entities: List[str], num_pairs: int = 10) -> List[Dict[str, str]]`：
   - 基于提取的实体构建问答对。
   - 参数：`entities`（实体列表），`num_pairs`（要生成的问答对数量，默认为 10）。
   - 随机选择实体和问题模板，构建问题并使用`gpt-4o-mini`模型生成答案。
   - 返回包含问题和答案的字典列表。
6. `save_to_json(self, entities: List[str], qa_pairs: List[Dict[str, str]], filename: str)`：
   - 将实体和问答对保存为 JSON 文件。
   - 参数：`entities`（实体列表），`qa_pairs`（问答对列表），`filename`（要保存的文件名）。

##### 注意事项：

1. **图像路径设置**：在调用`extract_entities`和`build_qa_pairs`方法之前，必须通过`set_topology_image`方法设置拓扑图像路径。
2. **图像编码**：`extract_entities`和`build_qa_pairs`方法中使用了`encode_image`函数对图像进行 base64 编码，确保该函数已正确实现。
3. **问答对生成**：`build_qa_pairs`方法默认生成 10 个问答对，可以根据需要调整`num_pairs`参数。
4. **JSON 文件保存**：`save_to_json`方法将实体和问答对保存为 JSON 文件，确保指定的`filename`路径可写。
5. **随机性**：`build_qa_pairs`方法中使用了随机选择实体和问题模板，这可能导致每次生成的问答对不同。


In [6]:
class Cmanager:
    def __init__(self, api_base: str, api_key: str):
        self.client = self.get_client(api_base, api_key)
        self.topology_image_path = ""

    @staticmethod
    def get_client(base_url: str, api_key: str) -> OpenAI:
        """Initialize and return the API client."""
        return OpenAI(base_url=base_url, api_key=api_key)

    def set_topology_image(self, image_path: str):
        """Set the path for the topology image."""
        self.topology_image_path = image_path

    # STEP 1: Extract entities from the topology image
    def extract_entities(self) -> List[str]:
        """Extract named core network elements from the topology image."""
        if not self.topology_image_path:
            raise ValueError("Topology image path is not set.")

        base64_image = encode_image(self.topology_image_path)
        content_system = (
            "你是一位网络拓扑实体提取专家，专注于识别命名的关键网络元素。"
            "你将收到一张描绘网络拓扑的图像。"
            "你的任务是从图像中识别出**命名的**核心网络元素。"
            "从命名的核心网络元素中提取最多八个（**八个(8)最重要的命名核心网络元素**）。"
            "如果命名的核心网络元素少于八个，则输出所有这些元素。"
            "如果图像中没有至少一个命名的核心网络元素，则输出一个空列表：`[]`。"
            "1. 核心网络元素："
            "  - 核心网络元素是像路由器、交换机、服务器、防火墙等具有特定名称的必要组件。"
            "2. 输出格式："
            "  - 输出核心网络元素的**名称**（而不是类型），以字符串列表的形式呈现。请勿使用其他格式。"
            "  - 不要包含任何解释、分析或附加信息。"
            f"  - 对于具有命名元素的图像，示例输出：['R1', 'SW1', 'ServerA']"
            f"  - 对于没有命名元素的图像，示例输出：[]"
            "3. 重要说明："
            "  - **仅关注命名实体。无名元素应被忽略。**"
            "  - 最多提取**八个**核心网络元素名称（不同名称）。"
            "  - 输出提取的名称**与图像中出现的一致**。"
            "  - 确保输出为有效的列表。"
        )

        completion = self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": content_system},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "拓扑结构图"},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            },
                        },
                    ],
                },
            ],
            temperature=0.2,
        )

        response_content = completion.choices[0].message.content.strip()
        try:
            # Parse the response content to extract entities
            list_str = response_content[2:-2]
            entities = [entity.strip("'") for entity in list_str.split(", ")]
            if not all(isinstance(entity, str) for entity in entities):
                raise ValueError("Entities are not in the correct format.")
            return entities
        except Exception as e:
            print(f"Error processing entities: {e}")
            return []

    # STEP 2: Generate QA pairs based on extracted entities
    def build_qa_pairs(
        self, entities: List[str], num_pairs: int = 5
    ) -> List[Dict[str, str]]:
        """Generate question-answer pairs based on the provided entities."""
        qa_pairs = []
        base64_image = encode_image(self.topology_image_path)

        question_templates = [
            "请描述整体网络拓扑类型。它是星形、总线、环形、网状、树形还是混合拓扑？如果是混合拓扑，请具体说明组合方式。",
            "请指定网络拓扑图中的节点总数。随后，基于这个节点总数，列出网络拓扑中的主要节点，包括它们的具体名称或标识符，并简要描述每个主要节点的功能（如果主要节点没有名称，请根据拓扑图为它们分配名称）。",
            "请详细描述主要节点之间的连接，包括连接类型（例如，有线、无线、光纤）和连接关系（例如，一对一、一对多、多对多）。",
            "在此拓扑中，哪个节点是中心节点？为什么该节点被视为中心节点？（如果中心节点有名称，请根据拓扑图分配一个名称。）",
            "请简要总结网络拓扑，控制字数在以下限制内：少于10个节点时不超过100个字，10到15个节点时不超过200个字，超过15个节点时不超过300个字。总结应包括以下内容：\n\n*   **整体架构：** 星形、总线、环形、网状、树形或混合。\n*   **区域/设备描述：** 描述每个主要区域或设备（例如，服务器、路由器、交换机、客户端等），包括它们的数量、类型和主要目的。\n*   **连接方式：** 描述区域/设备之间的连接类型（例如，有线、无线、光纤）和连接关系（哪些节点相互连接）。\n*   **数据流：** 概述数据或控制信息在网络中的流动方式。\n*   **网络目标：** 总结网络架构的主要设计目标和功能，例如数据传输、资源共享、安全访问等。\n\n使用以下结构进行总结：\n\n该网络是一个[整体架构]拓扑。它由几个相互连接的区域和设备组成。首先，[区域/设备名称1]包括[数量][设备类型]，并通过[连接方式]相连。该区域的主要目的是[区域/设备1的目的]。它通过[连接类型]与[区域/设备名称2]相连，促进数据和控制流。[区域/设备名称2]由[数量][设备类型]组成，作为[区域/设备2的目的]。此外，[区域/设备名称2]还连接到其他区域/设备，如[区域/设备名称3]，形成整体网络结构。区域/设备之间的连接还包括[连接类型，例如，防火墙、VPN连接]，提供[连接的目的]。该网络旨在实现[整体网络目的，例如，高效的数据处理、安全的远程访问]。",
        ]

        max_pairs = min(num_pairs, len(question_templates))

        for i in range(max_pairs):
            question_template = question_templates[i]
            question = question_template

            content_system = (
                "你是一位网络拓扑专家助手。"
                "你的任务是根据提供的图像和关键节点实体列表回答有关网络拓扑的问题。"
                "指示："
                "1. 仔细检查拓扑图像及相应的关键节点实体：理解关键节点实体之间的连接和关系，并使用实体名称来识别组件。"
                "2. 准确简洁地回答问题：提供清晰直接的答案，不做假设或引入外部信息。"
                "3. 将你的答案输出为字符串。"
                "拓扑图像："
                "实体："
                f"{entities} "
                "问题："
                f"{question}"
            )

            completion = self.client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": content_system},
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": "拓扑结构图"},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                },
                            },
                            {"type": "text", "text": f"{question}"},
                        ],
                    },
                ],
                temperature=0.2,
                max_tokens=4095,
            )
            answer = completion.choices[0].message.content.strip()
            qa_pairs.append({"question": question, "answer": answer})

        return qa_pairs

    def save_to_json(
        self, entities: List[str], qa_pairs: List[Dict[str, str]], filename: str
    ):
        """Save the extracted entities and QA pairs to a JSON file."""
        data = {"entities": entities, "qa_pairs": qa_pairs}
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

初始化 Cmanager 实例并设置网络拓扑图像路径


In [7]:
api_base = "https://neudm.zeabur.app/v1"
api_key = "sk-T05m0OqxOgKUjErs8c231e1c02E24573A17977F5E839E91c"
cmanager = Cmanager(api_base=api_base, api_key=api_key)

In [8]:
sub_paths = ["normal"]
data_path = "data/images/"
res_path = "data/result/final_version/"

for sub in sub_paths:
    topology_image_path = os.path.join(data_path, sub)

    # 列出目录下的所有文件
    files = [
        filename
        for filename in os.listdir(topology_image_path)
        if os.path.isfile(os.path.join(topology_image_path, filename))
    ]

    # 只处理前50张照片
    for i, filename in enumerate(files[:50]):
        # 构建完整的文件路径
        file_path = os.path.join(topology_image_path, filename)
        cmanager.set_topology_image(file_path)
        try:
            entities = cmanager.extract_entities()
            if entities:
                print("Extracted network element entities:", entities)
                qa_pairs = cmanager.build_qa_pairs(entities)
                print("Built QA pairs:", qa_pairs)

                # 构建结果保存的目录路径
                result_dir = os.path.join(res_path, sub)
                if not os.path.exists(result_dir):
                    os.makedirs(result_dir)

                # 构建结果文件的完整路径
                image_name_without_ext = os.path.splitext(filename)[0]
                output_filename = f"{image_name_without_ext}.json"
                output_file_path = os.path.join(result_dir, output_filename)

                cmanager.save_to_json(entities, qa_pairs, output_file_path)
                print(f"Results saved to {output_file_path}")
        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

Extracted network element entities: ['Internet', 'AC1', 'Agg-S1', 'Acc-S1', 'AP1', 'AP2', 'Acc-S2', 'FTP Server']
Built QA pairs: [{'question': '请描述整体网络拓扑类型。它是星形、总线、环形、网状、树形还是混合拓扑？如果是混合拓扑，请具体说明组合方式。', 'answer': '该网络拓扑是混合拓扑。它结合了星形和树形拓扑的特点。中心的Agg-S1作为星形拓扑的核心节点，连接到多个接入交换机（如Acc-S1, Acc-S2, Acc-S3, Acc-S4），每个接入交换机又连接到不同的设备和部门，形成树形结构。'}, {'question': '请指定网络拓扑图中的节点总数。随后，基于这个节点总数，列出网络拓扑中的主要节点，包括它们的具体名称或标识符，并简要描述每个主要节点的功能（如果主要节点没有名称，请根据拓扑图为它们分配名称）。', 'answer': '网络拓扑图中的节点总数为 12。主要节点包括：\n\n1. **Internet**: 提供外部网络连接。\n2. **AC1**: 负责无线接入控制。\n3. **Agg-S1**: 聚合交换机，用于连接和管理多个接入交换机。\n4. **Acc-S1**: 接入交换机，连接AP1和AP2，提供接待中心的访客网络接入。\n5. **AP1**: 无线接入点，为接待中心提供无线网络。\n6. **AP2**: 无线接入点，为接待中心提供无线网络。\n7. **Acc-S2**: 接入交换机，连接FTP服务器和打印机，服务于研发部门。\n8. **FTP Server**: 提供文件传输服务。\n9. **Acc-S3**: 接入交换机，连接打印机，服务于市场部门。\n10. **Acc-S4**: 接入交换机，连接打印机，服务于行政部门。\n11. **Printer (R&D)**: 研发部门的打印设备。\n12. **Printer (Marketing & Administration)**: 分别为市场和行政部门的打印设备。'}, {'question': '请详细描述主要节点之间的连接，包括连接类型（例如，有线、无线、光纤）和连接关系（例如，一对一、一对多、多

In [9]:
# sub_paths = ["normal"]
# data_path = "/home/leonz/topo2text/data/test_data/"
# res_path = "data/result/test_data/"

# for sub in sub_paths:
#     topology_image_path = os.path.join(data_path, sub)

#     # 列出目录下的所有文件
#     files = [
#         filename
#         for filename in os.listdir(topology_image_path)
#         if os.path.isfile(os.path.join(topology_image_path, filename))
#     ]

#     # 只处理从第51张到最后一张的照片（索引从0开始，索引50对应第51张）
#     for filename in files[50:]:  # 从索引50开始选择
#         # 构建完整的文件路径
#         file_path = os.path.join(topology_image_path, filename)
#         cmanager.set_topology_image(file_path)
#         try:
#             entities = cmanager.extract_entities()
#             if entities:
#                 print("Extracted network element entities:", entities)
#                 qa_pairs = cmanager.build_qa_pairs(entities)
#                 print("Built QA pairs:", qa_pairs)

#                 # 构建结果保存的目录路径
#                 result_dir = os.path.join(res_path, sub)
#                 if not os.path.exists(result_dir):
#                     os.makedirs(result_dir)

#                 # 构建结果文件的完整路径
#                 image_name_without_ext = os.path.splitext(filename)[0]
#                 output_filename = f"{image_name_without_ext}.json"
#                 output_file_path = os.path.join(result_dir, output_filename)

#                 cmanager.save_to_json(entities, qa_pairs, output_file_path)
#                 print(f"Results saved to {output_file_path}")
#         except Exception as e:
#             print(f"Error processing file {file_path}: {e}")